The notebook illustrates the difference between min-max, percentile and mean quantization types for two cases - tensor without outlier, tensor with outlier

In [55]:
import torch

LOW = -50
HIGH = 150
SIZE = 20
params = (HIGH - LOW) * torch.rand((SIZE, 1)) + LOW

In [56]:
params_no_outlier = params.clone()
params_outlier = params.clone()
params_outlier[0] = 1000
params_outlier[-1] = -1000

In [57]:
# minmax quantization
def assymetric_quantization_minmax(tensor : torch.Tensor, n_bits : int):
    """
    Assymetric quantization of a tensor
    """
    lower_bound = torch.min(tensor)
    upper_bound = torch.max(tensor)
    scale = (upper_bound - lower_bound) / (2 ** n_bits - 1)
    zero = -1.0 * torch.round(lower_bound / scale)
    quantized = torch.clamp(torch.round(tensor/scale + zero), 0, 2**n_bits-1).to(torch.int32)
    return quantized,scale,zero

def symmetric_quantization_minmax(tensor, n_bits):
    """
    Symmetric quantization of a tensor
    """
    upper_bound = torch.max(torch.abs(tensor))
    scale = upper_bound / (2**(n_bits-1)-1)
    quantized = torch.clamp(torch.round(tensor/scale), -2**(n_bits-1), 2**(n_bits-1)-1).to(torch.int32)
    return quantized, scale

In [58]:
# percentile quantization
def assymetric_quantization_percentile(tensor : torch.Tensor, n_bits : int, percentile : float =99.99):
    lower_bound = torch.quantile(tensor, (100-percentile)/100, interpolation='nearest')
    upper_bound = torch.quantile(tensor, percentile/100)
    scale = (upper_bound - lower_bound) / (2 ** n_bits - 1)
    zero = -1.0 * torch.round(lower_bound / scale)
    quantized = torch.clamp(torch.round(tensor/scale + zero),0,2**n_bits-1).to(torch.int32)
    return quantized,scale,zero

def symmetric_quantization_percentile(tensor, n_bits, percent : float = 99.99):
    """
    Symmetric quantization of a tensor
    """
    upper_bound = torch.quantile(torch.abs(tensor),percent/100, interpolation='nearest')
    scale = upper_bound / (2**(n_bits-1)-1)
    quantized = torch.clamp(torch.round(tensor/scale), -2**(n_bits-1), 2**(n_bits-1)-1).to(torch.int32)
    return quantized, scale

In [59]:
def asymmetric_dequantization(quantized, scale, zero):
    return (quantized - zero) * scale

def symmetric_dequantization(quantized, scale):
    return quantized * scale

def quantization_error(original, quantized):
    return torch.mean((original-quantized)**2)

In [60]:
# no outlier
# Asymmetric quantization - minmax
assymetric_q_mm, scale_a_mm, zero_a_mm = assymetric_quantization_minmax(params_no_outlier, 8)
# Symmetric quantization - minmax
symmetric_q_mm, scale_s_mm = symmetric_quantization_minmax(params_no_outlier, 8)
# Asymmetric quantization - percentile
assymetric_q_perc, scale_a_perc, zero_a_perc = assymetric_quantization_percentile(params_no_outlier, 8)
# Symmetric quantization - percentile
symmetric_q_perc, scale_s_perc = symmetric_quantization_percentile(params_no_outlier, 8)


In [61]:
params_no_outlier_dequant_amm = asymmetric_dequantization(assymetric_q_mm, scale_a_mm, zero_a_mm)
params_no_outlier_dequant_smm = symmetric_dequantization(symmetric_q_mm, scale_s_mm)
params_no_outlier_dequant_aperc = asymmetric_dequantization(assymetric_q_perc, scale_a_perc, zero_a_perc)
params_no_outlier_dequant_sperc = symmetric_dequantization(symmetric_q_perc, scale_s_perc)


In [62]:
# Errors
print(quantization_error(params_no_outlier, params_no_outlier_dequant_amm))
print(quantization_error(params_no_outlier, params_no_outlier_dequant_smm))
print(quantization_error(params_no_outlier, params_no_outlier_dequant_aperc))
print(quantization_error(params_no_outlier, params_no_outlier_dequant_sperc))

tensor(0.0412)
tensor(0.0675)
tensor(0.0423)
tensor(0.0675)


With no outlier, we see that both percentile and minmax quantizations offer similar quantization errors. in both cases, asymmetric is better than symmetric

In [63]:
# Outlier

# Asymmetric quantization - minmax
assymetric_q_mm, scale_a_mm, zero_a_mm = assymetric_quantization_minmax(params_outlier, 8)
# Symmetric quantization - minmax
symmetric_q_mm, scale_s_mm = symmetric_quantization_minmax(params_outlier, 8)
# Asymmetric quantization - percentile
assymetric_q_perc, scale_a_perc, zero_a_perc = assymetric_quantization_percentile(params_outlier, 8)
# Symmetric quantization - percentile
symmetric_q_perc, scale_s_perc = symmetric_quantization_percentile(params_outlier, 8)


In [64]:
params_outlier_dequant_amm = asymmetric_dequantization(assymetric_q_mm, scale_a_mm, zero_a_mm)
params_outlier_dequant_smm = symmetric_dequantization(symmetric_q_mm, scale_s_mm)
params_outlier_dequant_aperc = asymmetric_dequantization(assymetric_q_perc, scale_a_perc, zero_a_perc)
params_outlier_dequant_sperc = symmetric_dequantization(symmetric_q_perc, scale_s_perc)

In [65]:
print(quantization_error(params_outlier, params_outlier_dequant_amm))
print(quantization_error(params_outlier, params_outlier_dequant_smm))
print(quantization_error(params_outlier, params_outlier_dequant_aperc))
print(quantization_error(params_outlier, params_outlier_dequant_sperc))

tensor(5.7131)
tensor(4.1740)
tensor(5.7169)
tensor(4.1740)


With 99.99 percentile, we see percentile quantization doing a bit better than min-max for a tensor with outlier. let is adjust percentile to 99.9 to see if it does better

In [66]:
assymetric_q_perc, scale_a_perc, zero_a_perc = assymetric_quantization_percentile(params_outlier, 8, 99.999)
# Symmetric quantization - percentile
symmetric_q_perc, scale_s_perc = symmetric_quantization_percentile(params_outlier, 8,99.999)

In [67]:
params_outlier_dequant_aperc = asymmetric_dequantization(assymetric_q_perc, scale_a_perc, zero_a_perc)
params_outlier_dequant_sperc = symmetric_dequantization(symmetric_q_perc, scale_s_perc)

In [68]:
print(quantization_error(params_outlier, params_outlier_dequant_aperc))
print(quantization_error(params_outlier, params_outlier_dequant_sperc))

tensor(5.7068)
tensor(4.1740)


There is also another type of quantization called mean square quantization. it basically calculates the scale and zero such that the mean square between dequantized and original tensor is reduced.

In [69]:
def compute_quant(x, scale, zero, n_bits = 8):
    return torch.clamp(torch.round(x/scale+zero), 0, 2**n_bits-1).to(torch.int32)



In [70]:
def compute_dequant(quant, scale, zero):
    return (quant - zero) * scale

In [71]:
def compute_loss(original, dequantized):
    return 0.5 * torch.mean((original-dequantized)**2)

In [76]:
def compute_gradZ(x, scale, zero, zero_del, n_bits, DELTA):
    quant = compute_quant(x, scale, zero, n_bits)
    quant_del = compute_quant(x, scale, zero_del, n_bits)
    dequant = compute_dequant(quant, scale, zero)
    dequant_del = compute_dequant(quant_del, scale, zero_del)
    loss = compute_loss(x, dequant)
    loss_del = compute_loss(x, dequant_del)
    return (loss_del - loss) / DELTA

In [81]:
def compute_gradsc(x, scale, scale_del, zero, n_bits, DELTA):
    quant = compute_quant(x, scale, zero, n_bits)
    quant_del = compute_quant(x, scale_del, zero, n_bits)
    dequant = compute_dequant(quant, scale, zero)
    dequant_del = compute_dequant(quant_del, scale_del, zero)
    loss = compute_loss(x, dequant)
    loss_del = compute_loss(x, dequant_del)
    return (loss_del - loss) / DELTA

In [87]:
N_BITS = 8

DELTA = 0.0001

# let us take min_max scale and zero as initial scale and zero
def asymmetric_quantdequant(x, n_bits, LR, N_ITER):
    # take minmax scale and zero as initial scale and zeros
    scale_0 = (torch.max(x) - torch.min(x)) / (2 ** n_bits - 1)
    zero_0 = -1.0 * torch.round(torch.min(x) / scale_0)
    scale = scale_0
    zero = zero_0
    for i in range(N_ITER):
        zero_del = zero + DELTA
        scale_del = scale + DELTA
        gradZ = compute_gradZ(x, scale, zero, zero_del, n_bits, DELTA)
        gradsc = compute_gradsc(x, scale, scale_del, zero, n_bits, DELTA)
        scale = scale - LR * gradsc
        zero = zero - LR * gradZ
    quant = compute_quant(x, scale, zero, n_bits)
    dequant = compute_dequant(quant, scale, zero)
    return dequant




In [88]:
dequant_MSE = asymmetric_quantdequant(params_no_outlier, 8, 0.01,100)

In [89]:
print("quantization error for MSE quantization on no outlier data = ",
quantization_error(params_no_outlier, dequant_MSE))

quantization error for MSE quantization on no outlier data =  tensor(0.0554)


In [90]:
dequant_MSE = asymmetric_quantdequant(params_outlier, 8, 0.01, 100)

In [91]:
print("quantization error for MSE quantization on outlier data = ",
quantization_error(params_outlier, dequant_MSE))

quantization error for MSE quantization on outlier data =  tensor(6.3240)


on outlier data, we still have a lot of quantization error, let us try more iterations

In [92]:
dequant_MSE = asymmetric_quantdequant(params_outlier, 8, 0.005, 1000)

In [93]:
print("quantization error for MSE quantization on outlier data = ",
quantization_error(params_outlier, dequant_MSE))

quantization error for MSE quantization on outlier data =  tensor(3.0782)


we do see a big diff now, which means MSE has a potential to find better scale and zero values, but it is expensive process

In [94]:
dequant_MSE = asymmetric_quantdequant(params_outlier, 8, 0.001, 10000)
print("quantization error for MSE quantization on outlier data = ",
quantization_error(params_outlier, dequant_MSE))

quantization error for MSE quantization on outlier data =  tensor(2.7704)
